# Make Reservation and Get Port Number

## How to make a reservation

- Step 1: Go to **Access Our QC-Setups**

![Go to Access Our QC-Setups](./step_01_02.png)

- Step 2: Select a Service and Click **Queue In**

![Select a Service and Click Queue In](./step_03.png)

- Step 3: **Choose a QC-Setup**, **select threads for the job**, **select periods for the reservation**.
  **Confirm the queue summary** for your period and billing, optionally **add a note** for the reservation, and click **Confirm**.

![Make a reservation](./step_04.png)

- Step 4: After making a reservation, you will be redirected to the **Start our services** page. You can check the **job status** and get the code to access job.

![Check job status and get the code](./step_05.png)

## How to get the port number

Take the code you copied from the **Start our services** page.
Run it when the reservation is in the **running** state.

In [ ]:
from qctss_client.client import QCTSSClient

client = QCTSSClient()
client.config

BackendConfig(
  channel_name=default,
  backend_url=http://10.21.19.201:80/,
  fastapi_url=http://10.21.19.201:80/,
  websocket_url=ws://10.21.19.201:80/,
  timeout=30,
  max_retries=3,
  retry_delay=5)

In [3]:
from qctss_client.exceptions import JobCreationError

In [ ]:
try:
    job_response = client.start_job(
        qc_setup_list=["Long Live ASQPU_DR0_OPX1000_2_1"],
        service_name="HPC Integration Development",
    )
    job_status = client.wait_until_running(job_response.job_id)
    access_port = job_status.port_number
    print(f"| Job is running. Port number: {access_port}")
except JobCreationError as e:
    print(f"| Failed to create job: {e}")
    print(f"| Please check whether it's your reservation time or not. ")
    print(f"| If it's your reservation time, please contact the support team.")
    access_port = None

HTTP 400 error: {"error":["No active reservation or special event for these QCSetups. Token-based access requires a special event (can_add_job=True)."],"details":{"availability":"QCSetup not available"}}


| Failed to create job: Message: Job creation failed: {"error":["No active reservation or special event for these QCSetups. Token-based access requires a special event (can_add_job=True)."],"details":{"availability":"QCSetup not available"}} | HTTP 400 | Code: CLIENT_ERROR | Backend: {"error":["No active reservation or special event for these QCSetups. Token-based access requires a special event (can_add_job=True)."],"details":{"availability":"QCSetup not available"}}
| Please check whether it's your reservation time or not. 
| If it's your reservation time, please contact the support team.


If you run the code when the reservation is not in the **running** state, you will get an error message like below:

```
No active reservation or special event for these QCSetups
```


## Access the machine with the port number

When you get the port number, you can use it to access the machine from **Quantum Machine** or **Qblox** during your reservation period.


### Quantum Machine

In [ ]:
from quam_libs.components import QuAM

machine = QuAM.load()
machine.network["port"] = access_port

### Qblox

In [ ]:
from qblox_instruments import Cluster

cluster = Cluster("cluster", "10.21.19.201", access_port)

## Close the job before the reservation period ends


In [ ]:
client.close_job(job_response.job_id)

## Cancel the reservation

You can cancel the reservation by the commands below.

In [ ]:
client.cancel_job(job_response.job_id)